# ODI to Databricks Migration: TRG_DEP_Load

**Source File:** `TRG_DEP_Load.txt`
**Conversion Timestamp:** 2024-07-30 12:00:00
**Description:** This notebook converts an ODI process to load data into the `TRG_DEP` target table from the `DEPARTMENTS` source table in the `HR` schema.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

This section defines and displays the ETL parameters used throughout the notebook.

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    '${ETL_PROC_WID}' AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Target Table Setup

This section handles the drop and creation of the target table `trg_dep`.

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: (Implicitly handled by drop/create logic)
DROP TABLE IF EXISTS workspace.hr.trg_dep;

In [ ]:
%sql
-- SCEN_TASK_NO in {20}: (Implicitly handled by drop/create logic)
CREATE TABLE workspace.hr.trg_dep (
    department_id   BIGINT,
    department_name STRING,
    manager_id      BIGINT,
    location_id     BIGINT
) USING DELTA;

## Data Loading

This section inserts data into the `trg_dep` target table.

In [ ]:
%sql
-- SCEN_TASK_NO in {30}: Insert data into TRG_DEP
INSERT INTO workspace.hr.trg_dep
  (
    department_id,
    department_name,
    manager_id,
    location_id
  )
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

## Validation

This section provides basic validation of the loaded data.

In [ ]:
%sql
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_dep;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_dep LIMIT 10;

## Conversion Notes and Manual Actions Required

- The original ODI source only contained a simple `INSERT` statement. Standard `DROP TABLE` and `CREATE TABLE` statements have been added for a typical full load scenario.
- Placeholder `dbutils` widgets for common ETL parameters have been included, although they are not directly used by the simple `INSERT` logic.
- Table and schema names have been converted to lowercase and prefixed with `workspace.` as per Databricks naming conventions.
- Oracle hints `/*+ APPEND PARALLEL */` have been removed as they are not applicable to Databricks Delta Lake.
- Inferred Databricks data types for the `CREATE TABLE` statement based on common Oracle HR schema definitions (`NUMBER(p,0)` -> `BIGINT`, `VARCHAR2` -> `STRING`).